# Gold v2 Reference Final — Luxia 보완

목적:
- 기존 `gold_v2_final` 246장 × 4 HF LVLM × 3 Reference 결과는 그대로 재사용
- 빠져 있던 `Luxia`만 `Normal / Positive / Normal+Positive` 3조건으로 추가
- 최종적으로 5 LVLM × 3 Reference = 15조건을 한 결과표로 합침

재실행 안전:
- Luxia prediction은 Drive 임시 폴더에 checkpoint 저장
- 기존 HF 결과를 덮어쓰지 않음
- 완료 후 기존 `gold_reference_v2` 결과에 Luxia만 merge


In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"

QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
BASELINE_FOLDER = "lvlm_insplad_results/gold_baseline_v2"
REFERENCE_FOLDER = "lvlm_insplad_results/gold_reference_v2"
TMP_FOLDER = "lvlm_insplad_results/gold_reference_v2/_luxia_finalize_tmp"

CONDITIONS = ["normal_reference", "positive_reference", "normal_positive_reference"]
HF_MODELS = ["qwen3_5_4b", "internvl3_5_8b", "minicpm_v_4_5", "llava_onevision_7b"]
ALL_MODELS = HF_MODELS + ["luxia"]
CHECKPOINT_EVERY = 1

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167


In [ ]:
import base64, csv, importlib.util, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
BASELINE_DIR = DRIVE_ROOT / BASELINE_FOLDER
RESULT_DIR = DRIVE_ROOT / REFERENCE_FOLDER
TMP_DIR = DRIVE_ROOT / TMP_FOLDER
RESULT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""))
    return subprocess.run(cmd, env=env, check=check)

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token

luxia_key = userdata.get("LUXIA_API_KEY")
if not luxia_key:
    raise RuntimeError("Colab Secret LUXIA_API_KEY가 필요합니다.")
os.environ["LUXIA_API_KEY"] = luxia_key

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.extraHeader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
    "GIT_TERMINAL_PROMPT": "0",
})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env

run([sys.executable,"-m","pip","install","-q",
     "transformers==5.14.0","accelerate==1.14.0","sentencepiece==0.2.2",
     "pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66"])
run([sys.executable,"-m","pip","install","-q","-e",f"{REPO_DIR}[dev]"])

print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())


In [ ]:
def find_unique_sized(filename, expected_size):
    matches = [p for p in DRIVE_ROOT.rglob(filename) if p.is_file() and p.stat().st_size == expected_size]
    if not matches:
        raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")
    if len(matches) > 1:
        print("동일 크기 후보가 여러 개라 첫 번째 사용:", matches[:3])
    return matches[0]

SUP_ZIP = find_unique_sized(SUP_ZIP_NAME, SUP_ZIP_SIZE)
UNSUP_ZIP = find_unique_sized(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE)
BASELINE_LEADERBOARD = BASELINE_DIR / "leaderboard.csv"

positive_candidates = [
    RESULT_DIR / "positive_reference_build" / "positive_reference_manifest.csv",
    RESULT_DIR / "positive_reference_manifest.csv",
]
POSITIVE_MANIFEST = next((p for p in positive_candidates if p.is_file()), None)
if POSITIVE_MANIFEST is None:
    found = list(RESULT_DIR.rglob("positive_reference_manifest.csv"))
    if not found:
        raise FileNotFoundError("positive_reference_manifest.csv")
    POSITIVE_MANIFEST = found[0]

for path in (QA_DIR, SUP_ZIP, UNSUP_ZIP, BASELINE_LEADERBOARD, POSITIVE_MANIFEST):
    if not Path(path).exists():
        raise FileNotFoundError(path)

import pandas as pd
existing_lb = pd.read_csv(RESULT_DIR / "leaderboard.csv")
hf_existing = existing_lb[
    existing_lb.model_key.isin(HF_MODELS) & existing_lb.condition.isin(CONDITIONS)
].drop_duplicates(["model_key","condition"])
if len(hf_existing) != 12:
    raise RuntimeError(f"기존 HF reference 12조건이 필요합니다: 현재 {len(hf_existing)}")
print("기존 HF reference:", len(hf_existing), "/ 12")
print("positive manifest:", POSITIVE_MANIFEST)


In [ ]:
# Luxia 3조건만 별도 임시 결과 폴더에서 실행.
# GITHUB_TOKEN은 이 subprocess에서 제거해서 Luxia-only 임시 결과가 최신 GitHub 결과를 덮지 않게 함.
luxia_env = os.environ.copy()
luxia_env.pop("GITHUB_TOKEN", None)

run([
    sys.executable, REPO_DIR / "scripts/run_gold_v2_reference_ablation.py",
    "--qa-dir", QA_DIR,
    "--sup-zip", SUP_ZIP,
    "--unsup-zip", UNSUP_ZIP,
    "--positive-manifest", POSITIVE_MANIFEST,
    "--out-dir", TMP_DIR,
    "--baseline-leaderboard", BASELINE_LEADERBOARD,
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--conditions", *CONDITIONS,
    "--models", "luxia",
], env=luxia_env)

luxia_lb = pd.read_csv(TMP_DIR / "leaderboard.csv")
luxia_done = luxia_lb[
    (luxia_lb.model_key == "luxia") & luxia_lb.condition.isin(CONDITIONS)
].drop_duplicates(["model_key","condition"])
if len(luxia_done) != 3:
    raise RuntimeError(f"Luxia reference 3조건 미완료: {len(luxia_done)}/3")
display(luxia_done)


In [ ]:
# 기존 4 HF + 새 Luxia를 합쳐 최종 15조건 생성.
import importlib.util

def read_csv_rows(path):
    if not path.is_file() or path.stat().st_size == 0:
        return []
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def write_csv_rows(path, rows):
    fields = []
    for row in rows:
        for k in row:
            if k not in fields:
                fields.append(k)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

def merge_rows(old_rows, new_rows, keys):
    replace = {tuple(str(r.get(k,"")) for k in keys) for r in new_rows}
    kept = [r for r in old_rows if tuple(str(r.get(k,"")) for k in keys) not in replace]
    return kept + new_rows

final_lb = merge_rows(
    read_csv_rows(RESULT_DIR / "leaderboard.csv"),
    read_csv_rows(TMP_DIR / "leaderboard.csv"),
    ("model_key","condition"),
)
final_asset = merge_rows(
    read_csv_rows(RESULT_DIR / "metrics_by_asset.csv"),
    read_csv_rows(TMP_DIR / "metrics_by_asset.csv"),
    ("model_key","condition","asset_category"),
)
write_csv_rows(RESULT_DIR / "leaderboard.csv", final_lb)
write_csv_rows(RESULT_DIR / "metrics_by_asset.csv", final_asset)

for condition in CONDITIONS:
    src = TMP_DIR / condition / "predictions" / "luxia.jsonl"
    dst = RESULT_DIR / condition / "predictions" / "luxia.jsonl"
    if not src.is_file():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

old_status = []
status_path = RESULT_DIR / "run_status.json"
if status_path.is_file():
    old_status = json.loads(status_path.read_text(encoding="utf-8"))
new_status = json.loads((TMP_DIR / "run_status.json").read_text(encoding="utf-8"))
luxia_keys = {(x.get("model_key"), x.get("condition")) for x in new_status}
old_status = [x for x in old_status if (x.get("model_key"), x.get("condition")) not in luxia_keys]
status_path.write_text(json.dumps(old_status + new_status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

spec = importlib.util.spec_from_file_location(
    "ref_runner", REPO_DIR / "scripts/run_gold_v2_reference_ablation.py"
)
ref_runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ref_runner)
comparison, deltas = ref_runner.build_ablation_comparison(
    read_csv_rows(RESULT_DIR / "leaderboard.csv"), BASELINE_LEADERBOARD
)
write_csv_rows(RESULT_DIR / "ablation_comparison.csv", comparison)
write_csv_rows(RESULT_DIR / "ablation_vs_baseline.csv", deltas)

final = pd.read_csv(RESULT_DIR / "leaderboard.csv")
coverage = final[
    final.model_key.isin(ALL_MODELS) & final.condition.isin(CONDITIONS)
].drop_duplicates(["model_key","condition"])
if len(coverage) != 15:
    raise RuntimeError(f"최종 reference coverage mismatch: {len(coverage)} != 15")

metadata = {
    "benchmark_version": "gold_v2_final",
    "gold_images": 246,
    "conditions": CONDITIONS,
    "models": ALL_MODELS,
    "expected_runs": 15,
    "status": "complete" if len(coverage) == 15 else "incomplete",
    "note": "Existing 4 HF runs reused; Luxia finalized separately and merged without rerunning HF.",
}
(RESULT_DIR / "run_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
display(coverage.sort_values(["condition","model_key"]))


In [ ]:
# 핵심 결과 GitHub publish
publish_files = [
    "leaderboard.csv","metrics_by_asset.csv",
    "ablation_comparison.csv","ablation_vs_baseline.csv",
    "reference_manifest.csv","positive_reference_summary.csv",
    "positive_reference_metadata.json","run_status.json","run_metadata.json",
]
run([
    sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
    "--out-dir", RESULT_DIR,
    "--repo", REPO,
    "--branch", REPO_REF,
    "--prefix", "gold_v2_reference",
    "--files", *[f for f in publish_files if (RESULT_DIR / f).is_file()],
], env=os.environ.copy())

print("완료:", RESULT_DIR)
print("GitHub: results/latest/gold_v2_reference/")


## 완료 후

ChatGPT에 `reference Luxia까지 결과 나왔어`라고 하면 됩니다.

다음은 `lvlm_gold_v2_vision_expert_full_final_colab.ipynb`를 실행합니다.
